<a href="https://colab.research.google.com/github/Akshayab055/AI-Document-Search-and-Knowledge-Retrieval/blob/main/Sentimental%20Analysis%20Of%20Students%20Feedback.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Module 1: Data Loading and Preprocessing**
**Description:**
       This module loads the raw student feedback dataset, performs basic text cleaning (lowercasing, punctuation removal, tokenization), and applies lemmatization.
It also removes stopwords (except "not" for sentiment context). The output is saved as a preprocessed CSV file for the next steps.

In [ ]:
# Module 1: Data Loading and Preprocessing

import pandas as pd
import nltk
import os
import re
from nltk.tokenize.casual import casual_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# NLTK setup
nltk.download('stopwords')
nltk.download('wordnet')

# Load dataset
df = pd.read_csv('/content/student_feedback_dataset (1).csv')
df.dropna(subset=['Feedback'], inplace=True)

# Preprocessing function
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = casual_tokenize(text)
    stop_words = set(stopwords.words('english'))
    stop_words.discard('not')
    tokens = [word for word in tokens if word not in stop_words]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

df['processed_feedback'] = df['Feedback'].apply(preprocess_text)

# Save result
df.to_csv('/content/preprocessed_feedback.csv', index=False)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


 **Module 2: Sentiment Analysis using VADER**
**Description:**
This module uses VADER (Valence Aware Dictionary and sEntiment Reasoner) to compute sentiment scores.
Feedback is classified into Positive, Negative, or Neutral based on compound score thresholds.
The results are stored in a new CSV for later modeling.



In [ ]:
# Module 2: Sentiment Analysis using VADER

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import pandas as pd

nltk.download('vader_lexicon')

df = pd.read_csv('/content/preprocessed_feedback.csv')
sid = SentimentIntensityAnalyzer()

# Sentiment Scoring
def get_sentiment_score(text):
    return sid.polarity_scores(str(text))['compound']

df['sentiment_score'] = df['processed_feedback'].apply(get_sentiment_score)
df['predicted_sentiment'] = df['sentiment_score'].apply(
    lambda x: 'Positive' if x > 0.05 else ('Negative' if x < -0.05 else 'Neutral')
)

# Save result
df.to_csv('/content/sentiment_analysis_results.csv', index=False)


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


**Module 3: POS Tagging and Analysis**
**Description:**
This module performs Part-of-Speech tagging using NLTK and analyzes patterns in tags based on sentiment.
It extracts frequent POS tags from both positive and negative feedback.
This helps in understanding the grammatical patterns associated with different sentiments.


In [ ]:
import nltk
import pandas as pd

# Download the necessary NLTK data packages, including 'averaged_perceptron_tagger'
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
#The line below downloads the resource to avoid the error
nltk.download('averaged_perceptron_tagger_eng')
# Download the 'punkt_tab' data package
nltk.download('punkt_tab')

df = pd.read_csv("/content/sentiment_analysis_results.csv")

def pos_tags(text):
    tokens = nltk.word_tokenize(text)
    return nltk.pos_tag(tokens)

df["pos_tags"] = df["processed_feedback"].apply(pos_tags)
df.to_csv("pos_tags.csv", index=False)
print("✅ Module 3 complete: POS tags saved.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


✅ Module 3 complete: POS tags saved.


 **Module 4: Model Training using Naive Bayes**
**Description:**
This module trains a Naive Bayes classifier using TF-IDF vectorized feedback data.
It evaluates the model on a test set and prints out accuracy and classification metrics.
This provides a baseline machine learning model for sentiment prediction.

In [ ]:
# Module 4: Model Training using Naive Bayes

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('/content/sentiment_analysis_results.csv')
df = df.dropna(subset=['processed_feedback', 'predicted_sentiment'])

X_train, X_test, y_train, y_test = train_test_split(
    df['processed_feedback'], df['predicted_sentiment'], test_size=0.2, random_state=42)

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train.astype(str))
X_test_tfidf = tfidf.transform(X_test.astype(str))

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


Accuracy: 1.0
Classification Report:
               precision    recall  f1-score   support

    Negative       1.00      1.00      1.00        36
     Neutral       1.00      1.00      1.00        62
    Positive       1.00      1.00      1.00       102

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [ ]:
import pandas as pd
import nltk
import numpy as np
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('vader_lexicon')

# Load dataset
df = pd.read_csv('/content/student_feedback_dataset (1).csv')  # Adjust path if needed
df.dropna(subset=['Feedback', 'Sentiment'], inplace=True)

# === Custom Transformers ===

# Select a single column from DataFrame
class TextSelector(BaseEstimator, TransformerMixin):
    def __init__(self, key):
        self.key = key
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[self.key]

# Extract sentiment features using VADER
class SentimentFeatures(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.analyzer = SentimentIntensityAnalyzer()
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        scores = [self.analyzer.polarity_scores(str(text)) for text in X]
        return np.array([[s['neg'], s['neu'], s['pos'], s['compound']] for s in scores])

# === Split data ===
X_train, X_test, y_train, y_test = train_test_split(
    df[['Feedback']], df['Sentiment'], test_size=0.2, random_state=42)

# === FeatureUnion: TF-IDF + Sentiment Features ===
features = FeatureUnion([
    ('tfidf', Pipeline([
        ('selector', TextSelector('Feedback')),
        ('tfidf', TfidfVectorizer())
    ])),
    ('sentiment', Pipeline([
        ('selector', TextSelector('Feedback')),
        ('sentiment', SentimentFeatures())
    ]))
])

# === Final pipeline ===
pipeline = Pipeline([
    ('features', features),
    ('classifier', LogisticRegression(max_iter=1000))
])

# === Train model ===
pipeline.fit(X_train, y_train)

# === Evaluate ===
y_pred = pipeline.predict(X_test)
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("✅ Classification Report:\n", classification_report(y_test, y_pred))

# === Prediction function ===
def predict_feedback(feedback):
    df_input = pd.DataFrame({'Feedback': [feedback]})
    prediction = pipeline.predict(df_input)[0]
    print(f"🔍 Feedback: {feedback}")
    print(f"📌 Predicted Sentiment: {prediction}")
    return prediction

# === Test cases ===
predict_feedback("the lectures are interesting")
predict_feedback("the lectures are not interesting")
predict_feedback("the labs were okay, but not great")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
/usr/local/lib/python3.11/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


✅ Accuracy: 1.0
✅ Classification Report:
               precision    recall  f1-score   support

    Negative       1.00      1.00      1.00        91
    Positive       1.00      1.00      1.00       109

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

🔍 Feedback: the lectures are interesting
📌 Predicted Sentiment: Positive
🔍 Feedback: the lectures are not interesting
📌 Predicted Sentiment: Negative
🔍 Feedback: the labs were okay, but not great
📌 Predicted Sentiment: Negative


/usr/local/lib/python3.11/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


'Negative'

In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.7 MB/s eta 0:00:00


Dashboard Creation With Chart

In [ ]:
import pandas as pd
import nltk
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from wordcloud import WordCloud

# --- NLTK Downloads ---
nltk.download('punkt')
nltk.download('vader_lexicon')
nltk.download('averaged_perceptron_tagger')

# --- Load Dataset ---
df = pd.read_csv('/content/preprocessed_feedback.csv')
df.dropna(subset=['Feedback', 'Sentiment'], inplace=True)

# --- Custom Transformers ---
class TextSelector(BaseEstimator, TransformerMixin):
    def __init__(self, key):
        self.key = key
    def fit(self, X, y=None): return self
    def transform(self, X): return X[self.key]

class SentimentFeatures(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.analyzer = SentimentIntensityAnalyzer()
    def fit(self, X, y=None): return self
    def transform(self, X):
        scores = [self.analyzer.polarity_scores(str(text)) for text in X]
        return np.array([[s['neg'], s['neu'], s['pos'], s['compound']] for s in scores])

class POSTagFeatures(BaseEstimator, TransformerMixin):
    def get_pos_counts(self, text):
        tokens = nltk.word_tokenize(text)
        tags = nltk.pos_tag(tokens)
        tag_fd = nltk.FreqDist(tag for (word, tag) in tags)
        return [
            tag_fd['NN'],  # Noun
            tag_fd['VB'],  # Verb
            tag_fd['JJ'],  # Adjective
            tag_fd['RB']   # Adverb
        ]
    def fit(self, X, y=None): return self
    def transform(self, X):
        return np.array([self.get_pos_counts(str(text)) for text in X])

# --- Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    df[['Feedback']], df['Sentiment'], test_size=0.2, random_state=42)

# --- Feature Union ---
features = FeatureUnion([
    ('tfidf', Pipeline([
        ('selector', TextSelector('Feedback')),
        ('tfidf', TfidfVectorizer())
    ])),
    ('sentiment', Pipeline([
        ('selector', TextSelector('Feedback')),
        ('sentiment', SentimentFeatures())
    ])),
    ('pos_tags', Pipeline([
        ('selector', TextSelector('Feedback')),
        ('pos', POSTagFeatures())
    ]))
])

# --- Final Pipeline ---
pipeline = Pipeline([
    ('features', features),
    ('classifier', LogisticRegression(max_iter=1000))
])

# --- Train Model ---
pipeline.fit(X_train, y_train)

# --- Evaluate Model ---
y_pred = pipeline.predict(X_test)
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("✅ Classification Report:\n", classification_report(y_test, y_pred))

# --- Sentiment Analyzer ---
vader_analyzer = SentimentIntensityAnalyzer()

def analyze_sentiment_percentage(text):
    scores = vader_analyzer.polarity_scores(text)
    positive = scores['pos'] * 100
    negative = scores['neg'] * 100
    neutral = scores['neu'] * 100
    compound = scores['compound']

    result = f"""
Sentiment Analysis:
---------------------
✅ Positive: {positive:.2f}%
✅ Negative: {negative:.2f}%
✅ Neutral : {neutral:.2f}%

Overall Compound Score: {compound:.2f}
(Above 0 = Positive, Below 0 = Negative)
"""
    return result, scores

# --- Plot Pie Chart ---
def plot_sentiment_pie(text):
    _, scores = analyze_sentiment_percentage(text)
    labels = ['Positive', 'Negative', 'Neutral']
    sizes = [scores['pos'], scores['neg'], scores['neu']]
    colors = ['#2ecc71', '#e74c3c', '#f1c40f']

    fig, ax = plt.subplots()
    ax.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=140)
    ax.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
    plt.title('Sentiment Distribution')
    plt.tight_layout()
    return fig

# --- Combined Prediction and Plot Function ---
def predict_feedback_with_plot(feedback):
    prediction = pipeline.predict(pd.DataFrame({'Feedback': [feedback]}))[0]
    sentiment_text, _ = analyze_sentiment_percentage(feedback)
    pie_chart = plot_sentiment_pie(feedback)
    return f"{sentiment_text}\nPredicted Sentiment: {prediction}", pie_chart

# --- Gradio Interface ---
interface = gr.Interface(
    fn=predict_feedback_with_plot,
    inputs=gr.Textbox(lines=4, placeholder="Type student feedback here..."),
    outputs=["text", "plot"],
    title="🎓 Student Feedback Sentiment Analyzer",
    description="Analyze student feedback, predict sentiment, and view Positive/Negative percentages along with a pie chart!",
)

interface.launch(share=True)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
/usr/local/lib/python3.11/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


✅ Accuracy: 1.0
✅ Classification Report:
               precision    recall  f1-score   support

    Negative       1.00      1.00      1.00        91
    Positive       1.00      1.00      1.00       109

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://93b37cbd31c4587b2e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
#2nd output not working properly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
import io
import gradio as gr
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from matplotlib.backends.backend_pdf import PdfPages

# --- NLTK Downloads ---
nltk.download('punkt')
nltk.download('vader_lexicon')
nltk.download('averaged_perceptron_tagger')

# --- Load Dataset ---
df = pd.read_csv('/content/preprocessed_feedback.csv')  # Change path as needed
df.dropna(subset=['Feedback', 'Sentiment'], inplace=True)

# --- Custom Transformers ---
class TextSelector(BaseEstimator, TransformerMixin):
    def __init__(self, key):
        self.key = key
    def fit(self, X, y=None): return self
    def transform(self, X): return X[self.key]

class SentimentFeatures(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.analyzer = SentimentIntensityAnalyzer()
    def fit(self, X, y=None): return self
    def transform(self, X):
        scores = [self.analyzer.polarity_scores(str(text)) for text in X]
        return np.array([[s['neg'], s['neu'], s['pos'], s['compound']] for s in scores])

class POSTagFeatures(BaseEstimator, TransformerMixin):
    def get_pos_counts(self, text):
        tokens = nltk.word_tokenize(text)
        tags = nltk.pos_tag(tokens)
        tag_fd = nltk.FreqDist(tag for (word, tag) in tags)
        return [
            tag_fd['NN'],  # Noun
            tag_fd['VB'],  # Verb
            tag_fd['JJ'],  # Adjective
            tag_fd['RB']   # Adverb
        ]
    def fit(self, X, y=None): return self
    def transform(self, X):
        return np.array([self.get_pos_counts(str(text)) for text in X])

# --- Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    df[['Feedback']], df['Sentiment'], test_size=0.2, random_state=42)

# --- Feature Union ---
features = FeatureUnion([
    ('tfidf', Pipeline([
        ('selector', TextSelector('Feedback')),
        ('tfidf', TfidfVectorizer())
    ])),
    ('sentiment', Pipeline([
        ('selector', TextSelector('Feedback')),
        ('sentiment', SentimentFeatures())
    ])),
    ('pos_tags', Pipeline([
        ('selector', TextSelector('Feedback')),
        ('pos', POSTagFeatures())
    ]))
])

# --- Final Pipeline ---
pipeline = Pipeline([
    ('features', features),
    ('classifier', LogisticRegression(max_iter=1000))
])

# --- Train Model ---
pipeline.fit(X_train, y_train)

# --- Evaluate Model ---
y_pred = pipeline.predict(X_test)
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("✅ Classification Report:\n", classification_report(y_test, y_pred))

# --- Sentiment Analyzer ---
vader_analyzer = SentimentIntensityAnalyzer()

def analyze_sentiment_percentage(text):
    scores = vader_analyzer.polarity_scores(text)
    positive = scores['pos'] * 100
    negative = scores['neg'] * 100
    neutral = scores['neu'] * 100
    compound = scores['compound']

    result = f"""
Sentiment Analysis:
---------------------
✅ Positive: {positive:.2f}%
✅ Negative: {negative:.2f}%
✅ Neutral : {neutral:.2f}%

Overall Compound Score: {compound:.2f}
(Above 0 = Positive, Below 0 = Negative)
"""
    return result, scores

def plot_sentiment_pie(text):
    _, scores = analyze_sentiment_percentage(text)
    labels = ['Positive', 'Negative', 'Neutral']
    sizes = [scores['pos'], scores['neg'], scores['neu']]
    colors = ['#2ecc71', '#e74c3c', '#f1c40f']

    fig, ax = plt.subplots()
    ax.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=140)
    ax.axis('equal')
    plt.title('Sentiment Distribution')
    plt.tight_layout()
    return fig

# --- Single Prediction Function ---
def predict_feedback_with_plot(feedback):
    prediction = pipeline.predict(pd.DataFrame({'Feedback': [feedback]}))[0]
    sentiment_text, _ = analyze_sentiment_percentage(feedback)
    pie_chart = plot_sentiment_pie(feedback)
    return f"{sentiment_text}\nPredicted Sentiment: {prediction}", pie_chart

# --- Batch Analysis Function with PDF/CSV Export ---
def batch_analyze_feedback(file):
    df_uploaded = pd.read_csv(file.name)

    if 'Feedback' not in df_uploaded.columns:
        return "❌ CSV must contain a 'Feedback' column.", None, None

    results = []
    pdf_buffer = io.BytesIO()

    with PdfPages(pdf_buffer) as pdf:
        for i, row in df_uploaded.iterrows():
            text = row['Feedback']
            prediction = pipeline.predict(pd.DataFrame({'Feedback': [text]}))[0]
            scores = vader_analyzer.polarity_scores(text)
            results.append({
                'Feedback': text,
                'Predicted Sentiment': prediction,
                'Positive': scores['pos'],
                'Negative': scores['neg'],
                'Neutral': scores['neu'],
                'Compound': scores['compound']
            })

            # Pie chart
            labels = ['Positive', 'Negative', 'Neutral']
            sizes = [scores['pos'], scores['neg'], scores['neu']]
            colors = ['#2ecc71', '#e74c3c', '#f1c40f']
            fig, ax = plt.subplots()
            ax.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=140)
            ax.set_title(f"Feedback {i+1}: Sentiment Pie")
            pdf.savefig(fig)
            plt.close(fig)

    # CSV buffer
    results_df = pd.DataFrame(results)
    csv_buffer = io.StringIO()
    results_df.to_csv(csv_buffer, index=False)
    csv_bytes = io.BytesIO(csv_buffer.getvalue().encode('utf-8'))

    pdf_buffer.seek(0)
    csv_bytes.seek(0)
    return "✅ Batch analysis complete!", csv_bytes, pdf_buffer

# --- Gradio Interfaces ---
single_interface = gr.Interface(
    fn=predict_feedback_with_plot,
    inputs=gr.Textbox(lines=4, placeholder="Type student feedback here..."),
    outputs=["text", "plot"],
    title="🎓 Single Feedback Sentiment Analyzer",
    description="Analyze student feedback, predict sentiment, and view sentiment pie chart."
)

batch_interface = gr.Interface(
    fn=batch_analyze_feedback,
    inputs=gr.File(label="Upload CSV File with 'Feedback' Column"),
    outputs=[
        "text",
        gr.File(label="📄 Download CSV Report"),
        gr.File(label="📊 Download PDF Report with Pie Charts")
    ],
    title="📥 Batch Feedback Analyzer",
    description="Upload a CSV with 'Feedback' column. Download sentiment predictions as CSV and PDF."
)

# --- Launch Tabbed Interface ---
gr.TabbedInterface(
    [single_interface, batch_interface],
    ["🔍 Single Feedback", "📊 Batch Upload & Report"]
).launch(share=True)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
/usr/local/lib/python3.11/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


✅ Accuracy: 1.0
✅ Classification Report:
               precision    recall  f1-score   support

    Negative       1.00      1.00      1.00        91
    Positive       1.00      1.00      1.00       109

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3ac7bd2fcf94110892.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
